In [37]:
#! pip install h2ogpte==1.4.10

In [21]:
from h2ogpte import H2OGPTE
import os
import ast

In [22]:
# variables neccessary for testing
qns_count = 1
prompts_folder_name = "prompts_4"

# parsing only works on list of list, sample as /prompts output
do_you_need_output_parsing = True

# if parsing required:
# Peizhi's original format [[x],[x,x,x,x],[x]] use 0
# Xiaoya's [x,[x,x,x,x],x] format use 1
desired_parsing_format = 1
 

# Client Creation

In [23]:
# This is the new GLOBAL API KEY
api_key = 'sk-s664ThZtgjVvGG3Fl1mGN9gOVnfpg85dZBwMWQhb8YBqXbOT'

In [24]:
client = H2OGPTE(address='https://h2ogpte.genai.h2o.ai', api_key=api_key)
collection_names = [item.name for item in client.list_recent_collections(0, 1000)]
collection_names

Please install the correct version of H2OGPTE with `pip install h2ogpte==1.4.10`.
You can enable strict version checking by passing strict_version_check=True.


['The Holocaust',
 'Hundred Years War',
 'War and the Other',
 'The Vikings',
 'GEH1079']

In [25]:
client.get_llms()

[{'base_model': 'mistralai/Mixtral-8x7B-Instruct-v0.1',
  'prompt_type': 'mixtral',
  'prompt_dict': {'promptA': '<<SYS>>\nYou are an AI that follows instructions extremely well and as helpful as possible.\n<</SYS>>\n\n',
   'promptB': '<<SYS>>\nYou are an AI that follows instructions extremely well and as helpful as possible.\n<</SYS>>\n\n',
   'PreInstruct': '<s> [INST] ',
   'PreInput': None,
   'PreResponse': '[/INST]',
   'terminate_response': ['[INST]', '</s>'],
   'chat_sep': ' ',
   'chat_turn_sep': '</s> ',
   'humanstr': '[INST]',
   'botstr': '[/INST]',
   'generates_leading_space': False,
   'system_prompt': 'You are an AI that follows instructions extremely well and as helpful as possible.',
   'can_handle_system_prompt': True},
  'load_8bit': False,
  'load_4bit': False,
  'low_bit_mode': 1,
  'load_half': False,
  'use_flash_attention_2': False,
  'load_gptq': '',
  'load_awq': '',
  'load_exllama': False,
  'use_safetensors': False,
  'revision': None,
  'use_gpu_id': F

In [26]:
curr_name = "The Holocaust"

curr_collection_id = [item.id for item in client.list_recent_collections(0, 1000) if item.name == curr_name][0]
curr_collection_id

'aa188420-d726-4fdc-b937-8eb5adfda57c'

In [27]:
# Initiate chat session on selected curr_collection_id
chat_session_id = client.create_chat_session(curr_collection_id)

## Import the Prompts

Enter the folder name with prompts txt files inside. Results are stored in folder /prompts performances. 10 questions are generated each time for evaluation purposes.
* pre_prompt_query.txt
* prompt_query.txt
* system_prompt.txt

In [28]:
# cwd = os.getcwd()
# print(cwd)

In [29]:
# with open(cwd  + '/prompts_global_1/system_prompt.txt', 'r') as file:
#     system_prompt = file.read()

# with open('prompts_global_1/pre_prompt_query.txt', 'r') as file:
#     pre_prompt_query = file.read()

# with open('prompts_global_1/prompt_query.txt', 'r') as file:
#     prompt_query = file.read()

In [30]:
with open(f'./{prompts_folder_name}/system_prompt.txt', 'r') as file:
    system_prompt = file.read()

with open(f'./{prompts_folder_name}/pre_prompt_query.txt', 'r') as file:
    pre_prompt_query = file.read()

with open(f'./{prompts_folder_name}/prompt_query.txt', 'r') as file:
    prompt_query = file.read()

# Client Query

In [31]:
query = f'Generate {qns_count} MCQ questions based on given documents.'
print(query)

Generate 1 MCQ questions based on given documents.


In [32]:
with client.connect(chat_session_id) as session:
    reply = session.query(
        message = query,
        system_prompt = system_prompt,
        pre_prompt_query = pre_prompt_query,
        prompt_query = prompt_query,
        timeout=60,
        llm='gpt-35-turbo-1106',
        llm_args={"temperature": 0.9},
    )

print(reply.content)

[
  ["Why did Nazi Germany continue, even when it started to lose a total war against stronger enemy states?", ["To achieve the complete defeat of the enemy physical power", "To restore history to the 'right kind of people'", "To take revenge for the Treaty of Versailles", "To protest against the Great War and the Treaty of Versailles"], "To achieve the complete defeat of the enemy physical power"]
]


In [33]:
with client.connect(chat_session_id) as session:
    reply = session.query(
        message = query,
        system_prompt = system_prompt,
        pre_prompt_query = pre_prompt_query,
        prompt_query = prompt_query,
        timeout=60,
        llm='gpt-35-turbo-1106', 
        llm_args={"temperature": 0.9},
    )

print(reply.content)

[
["According to the information, why did Nazi Germany continue the war against the Jewish people?",["To achieve complete defeat of enemy physical power","For special non-military purposes","To remove the Jews from German national life","To restore history to the 'right kind of people'"],"To achieve complete defeat of enemy physical power"],
["What was the objective of the Final Solution?",["To achieve complete defeat of enemy physical power","To achieve a systematic project of total war","Genocide","To escalate the level of violence"],"Genocide"],
["What determined whether someone was considered a Jew according to the Nuremberg Laws?",["Race","Religion","Customs","All of the above"],"All of the above"]
]


Save the questions together with query message.

Saving in a subfolder is not working properly, nor did the shutil.move function. Move the files mannually afterwards.

In [34]:
import ast
import shutil

# Open the file in write mode and write the data
def record_output_parsing_lists(file_name, content, additional_info_prefix = None, format_option = 0):
    with open(file_name, "w",encoding="utf-8") as file:
        if additional_info_prefix:
            file.write(additional_info_prefix + '\n\n')
        
        for item in content:
            # Write the question
            file.write(''.join(item[0]) + '\n')

            # Write the options
            for option in item[1]:
                file.write(option + '\n')

            if format_option == 0 :
                # Write the correct answer
                file.write(f"[{item[2][0]}]\n\n")
            elif format_option == 1:
                file.write(''.join(item[2])+ '\n\n')

# This function for formated output, straight write to txt files.
def record_output_without_parsing(file_name, content, additional_info_prefix):
    with open(file_name, "w", encoding="utf-8") as file:
        if additional_info_prefix:
            file.write(additional_info_prefix + '\n\n')

        file.write(content)
        


In [35]:

# Define the filename
filename = f"output_using_set_{prompts_folder_name}.txt"

if do_you_need_output_parsing:
    try:
        content = ast.literal_eval(reply.content)
        record_output_parsing_lists(filename, content=content, additional_info_prefix=query, format_option= desired_parsing_format)
    except SyntaxError:
        record_output_without_parsing(filename, content=reply.content, additional_info_prefix= query)
else:
    record_output_without_parsing(filename, content=reply.content, additional_info_prefix= query)

    
# Move the file to the "prompts performances" folder
# shutil.move(filename, f"/prompts performances/{filename}")
